# 第12回：3人ゲームのコアを三角形上に描く

ベンチャー企業の起業ゲームについて，$x_A+x_B+x_C=240$ を満たす配分を正三角形上の点として表す．

- 薄い青：パレート最適な配分全体
- 灰色：個人合理性を満たす配分
- 緑色：すべての提携合理性を満たすコア

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib import font_manager
from matplotlib.patches import Polygon

preferred_fonts = [
    "Hiragino Sans",
    "Yu Gothic",
    "Noto Sans CJK JP",
    "IPAexGothic",
]
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
for font_name in preferred_fonts:
    if font_name in available_fonts:
        plt.rcParams["font.family"] = font_name
        break

plt.rcParams["axes.unicode_minus"] = False


## 配分から平面座標への変換

正三角形の頂点を，Aが全額を受け取る配分，Bが全額を受け取る配分，Cが全額を受け取る配分に対応させる．各配分は3頂点の加重平均として平面上へ写される．

In [ ]:
TOTAL = 240

triangle_vertices = {
    "A": np.array([0.5, np.sqrt(3) / 2]),
    "B": np.array([0.0, 0.0]),
    "C": np.array([1.0, 0.0]),
}


def to_cartesian(allocation, total=TOTAL):
    """配分 (x_A, x_B, x_C) を正三角形上の座標へ変換する．"""
    x_a, x_b, x_c = allocation
    if not np.isclose(x_a + x_b + x_c, total):
        raise ValueError("配分の合計が全員提携の価値と一致しません．")

    return (
        (x_a / total) * triangle_vertices["A"]
        + (x_b / total) * triangle_vertices["B"]
        + (x_c / total) * triangle_vertices["C"]
    )


def polygon_coordinates(allocations):
    return np.array([to_cartesian(allocation) for allocation in allocations])


## 配分集合とコアの描画

個人合理的な配分は $x_A\geq 60$，$x_B\geq 40$，$x_C\geq 20$ を満たす．コアではさらに $x_A+x_B\geq 200$，$x_A+x_C\geq 150$，$x_B+x_C\geq 100$ を満たす．

In [ ]:
all_allocations = [(240, 0, 0), (0, 240, 0), (0, 0, 240)]
individually_rational = [(180, 40, 20), (60, 160, 20), (60, 40, 140)]
core_vertices = [
    (110, 90, 40),
    (130, 90, 20),
    (140, 80, 20),
    (140, 60, 40),
]

fig, ax = plt.subplots(figsize=(9, 8), constrained_layout=True)

ax.add_patch(
    Polygon(
        polygon_coordinates(all_allocations),
        closed=True,
        facecolor="#D7E8F5",
        edgecolor="#2C5F8A",
        linewidth=2.2,
        label="パレート最適な配分",
    )
)
ax.add_patch(
    Polygon(
        polygon_coordinates(individually_rational),
        closed=True,
        facecolor="#B8B8B8",
        edgecolor="#666666",
        alpha=0.55,
        linewidth=1.8,
        label="個人合理的な配分",
    )
)
ax.add_patch(
    Polygon(
        polygon_coordinates(core_vertices),
        closed=True,
        facecolor="#59A14F",
        edgecolor="#246B24",
        alpha=0.82,
        linewidth=2.2,
        label="コア",
    )
)

core_label_offsets = {
    (110, 90, 40): (-105, -20),
    (130, 90, 20): (-100, -6),
    (140, 80, 20): (-100, 14),
    (140, 60, 40): (12, 14),
}

for allocation in core_vertices:
    point = to_cartesian(allocation)
    ax.scatter(*point, color="#174A17", s=36, zorder=5)
    ax.annotate(
        str(allocation),
        point,
        xytext=core_label_offsets[allocation],
        textcoords="offset points",
        fontsize=9,
    )

outside_core = (100, 80, 60)
outside_point = to_cartesian(outside_core)
ax.scatter(
    *outside_point,
    marker="X",
    color="#D62728",
    s=95,
    zorder=6,
    label="コア外の配分 (100, 80, 60)",
)

for player, vertex in triangle_vertices.items():
    offset = {"A": (0, 14), "B": (-46, -18), "C": (8, -18)}[player]
    allocation = {
        "A": "(240, 0, 0)",
        "B": "(0, 240, 0)",
        "C": "(0, 0, 240)",
    }[player]
    ax.annotate(
        f"{player}: {allocation}",
        vertex,
        xytext=offset,
        textcoords="offset points",
        fontsize=10,
        weight="bold",
    )

ax.set_xlim(-0.12, 1.12)
ax.set_ylim(-0.10, 0.99)
ax.set_aspect("equal")
ax.set_title("ベンチャー企業の起業ゲームにおけるコア")
ax.legend(loc="upper right", framealpha=0.96)
ax.axis("off")

output_path = Path("../figs/12/core_triangle.png")
output_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output_path, dpi=180, bbox_inches="tight")
plt.show()

print(f"保存先: {output_path}")
